# Notebook 02 — Model Development

NB02 trains four models and computes metrics. **It does not author a validation verdict** — that is NB03's job, and the notebook enforces this at Phase 16 by writing 'performance_summary_v2.json' without risk_score/risk_rating/recommendation fields and listing those in 'does_not_contain'. The reader should understand this is a deliberate separation of duties: the developer notebook produces evidence; the validator notebook produces the verdict.

In [1]:
# Notebook 02 — Model Development
# Version 2.1  [EDIT E-M02-VER] bumped from 2.0
# Changes vs 2.0:
#   E-M02-VER    version bump
#   E-M02-01     reads NB01 contract (data_quality_summary.json) — no defaults
#   E-M02-02     leakage features read from feature_inventory.csv (B1)
#   E-M02-03     emp_years guard raises explicit finding (A5)
#   E-M02-04     split-method exception disclosed in output, not silent (A4)
#   E-M02-05     tool prints computed metrics only — no validation opinion (A1)
#   E-M02-06     HL p-value diagnostic flags p ∈ {0.0000, 1.0000} (B5)
#   E-M02-07     every [CRITICAL]/[ERROR]/[WARNING] token has findings entry (C1)
#   E-M02-08     run_id / notebook_version on all artifacts
#   E-M02-09     performance_summary_v2.json is the sole contract for NB03
#
# NOTE: This notebook trains models and computes metrics. It does NOT author a validation verdict. That is NB03's job. This notebook's output —
# performance_summary_v2.json — is the machine contract NB03 reads. It deliberately contains no risk_score, risk_rating, or recommendation field.
# The assertion at Phase 16 enforces this.
# 
# What breaks if you change this notebook without updating downstream:
# - NB03 reads performance_summary_v2.json. Adding fields is safe; removing or renaming breaks NB03. The `does_not_contain` field is explicit — if
#   you accidentally add a verdict field, NB03 raises at Phase 3.
# - NB03 reads models/test_data.pkl, models/predictions.pkl, and the four model pickle files. Any change to their schema breaks NB03.
# - The findings register (findings_register_v2.csv) is read by NB03 and NB04. Finding IDs (F-02.01 etc.) are stable identifiers. Do not renumber.

### PHASE 1 - ENVIRONMENT & REPRODUCIBILITY

In [2]:
print("\n" + "=" * 80)
print("PHASE 1: ENVIRONMENT SETUP & REPRODUCIBILITY")
print("=" * 80)

# NOTE: Same version-bump discipline as NB01. This notebook's version appears in every artifact it produces (run_id / notebook_version
# columns on CSVs, fields in the JSON). A downstream reviewer can tell which version of NB02 produced a given model set.
NOTEBOOK_VERSION = "2.1"                                    # [EDIT E-M02-VER]

# NOTE: 42 is the project-wide seed from AS-001 §8. It is used for:
# - train_test_split(random_state=42)
# - StratifiedKFold(shuffle=True, random_state=42)
# - XGBClassifier(random_state=42), LightGBM, RandomForest
# If you change it in one place and not others, cross-validation results won't be reproducible across runs of the same code.
RANDOM_STATE = 42

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
import json
import sys
import logging
import hashlib
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

# NOTE: sklearn's Pipeline is used for LogisticRegression to bundle the imputer + classifier. This means predict_proba on the pipeline handles
# any residual NaN in X_train or X_val automatically. The other three models (XGBoost, RF, LightGBM) are trained directly, which is fine
# because Phase 5 imputes all NaN before the split.
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, average_precision_score,
    brier_score_loss, log_loss
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.calibration import calibration_curve
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from scipy.stats import chi2

import xgboost as xgb
import lightgbm as lgb

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 50)

os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("outputs/figures", exist_ok=True)
os.makedirs("outputs/reports", exist_ok=True)


def setup_audit_logger(run_id=None):
    """Per-run audit logger.

    NOTE: Same pattern as NB01. Every run of this notebook gets a distinct run_id, and every artifact carries that run_id as a
    column or field. A reviewer can reconstruct which artifacts were produced by which run.
    """
    if run_id is None:
        run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    os.makedirs("logs", exist_ok=True)
    logger = logging.getLogger(f"lending_club_nb02_{run_id}")
    logger.setLevel(logging.INFO)
    if logger.hasHandlers():
        logger.handlers.clear()
    fh = logging.FileHandler(f"logs/model_dev_audit_trail_{run_id}.log", encoding="utf-8")
    fh.setFormatter(logging.Formatter(
        '%(asctime)s | %(levelname)s | %(message)s', datefmt='%Y-%m-%d %H:%M:%S'))
    logger.addHandler(fh)
    ch = logging.StreamHandler(sys.stdout)
    ch.setFormatter(fh.formatter)
    logger.addHandler(ch)
    return logger, run_id


def compute_file_hash(path):
    """SHA-256-16 of a file, chunked read.

    NOTE: Same function as NB01, kept local because notebooks are meant to be self-contained (a reviewer copies one and runs it in isolation).
    If you refactor into a shared module, replace with `from shared import compute_file_hash` and note the dependency in the edit log.
    """
    sha256 = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            sha256.update(chunk)
    return sha256.hexdigest()[:16]


def _write_json(obj, path):
    """Write a dict as JSON with hash logging.

    NOTE: Every JSON artifact carries provenance. The JSON files are the machine contracts. If you add a field, note it in the edit log so
    downstream readers (NB03, NB04) know. If you remove a field, you break downstream and must version-bump.
    """
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)
    audit_logger.info(f"Saved: {path} hash={compute_file_hash(path)}")


audit_logger, RUN_ID = setup_audit_logger()
audit_logger.info("=" * 60)
audit_logger.info(f"NOTEBOOK 02 v{NOTEBOOK_VERSION}: MODEL DEVELOPMENT")
audit_logger.info(f"RUN ID: {RUN_ID}")
audit_logger.info("=" * 60)

print(f"[OK] Project-wide random seed: {RANDOM_STATE}")
print(f"[OK] Run ID: {RUN_ID}  |  Notebook version: {NOTEBOOK_VERSION}")
np.random.seed(RANDOM_STATE)
audit_logger.info(f"Random seed set: {RANDOM_STATE}")


PHASE 1: ENVIRONMENT SETUP & REPRODUCIBILITY
2026-10-22 11:43:11 | INFO | ============================================================
2026-10-22 11:43:11 | INFO | NOTEBOOK 02 v2.1: MODEL DEVELOPMENT
2026-10-22 11:43:11 | INFO | RUN ID: 20261022_114311
2026-10-22 11:43:11 | INFO | ============================================================
[OK] Project-wide random seed: 42
[OK] Run ID: 20261022_114311  |  Notebook version: 2.1
2026-10-22 11:43:11 | INFO | Random seed set: 42


### PHASE 2: DATASET CAVEAT (REFERENCED)

In [3]:
# NOTE: The caveat is referenced, not repeated. The full caveat lives in NB01 Phase 1 and in the audit log. Repeating it here would create two
# sources of truth for the same disclaimer, which would drift. Referencing is the discipline.
print("\n" + "=" * 80)
print("PHASE 2: DATASET CAVEAT (REFERENCED)")
print("=" * 80)
print("""
This notebook uses the dataset described in NB01 (Phase 1 Dataset Caveat). Lending Club is a U.S. proxy for proprietary bank data. Not validated for Canadian portfolios (OSFI E-23 applicability gap). All conclusions are illustrative.
""")


PHASE 2: DATASET CAVEAT (REFERENCED)

This notebook uses the dataset described in NB01 (Phase 1 Dataset Caveat). Lending Club is a U.S. proxy for proprietary bank data. Not validated for Canadian portfolios (OSFI E-23 applicability gap). All conclusions are illustrative.



### PHASE 3: LOAD DATA & FEATURE INVENTORY

In [4]:
# [EDIT E-M02-01] Every NB01 artifact that NB02 needs is loaded here. If any is missing, raise — do not fall back to hardcoded defaults.

# NOTE: This phase is a *contract enforcement point*. NB02 refuses to run if NB01's outputs are missing, because computing from a partial input
# set would silently produce results that look complete but aren't.
# 
# Specifically:
# - feature_inventory.csv: required. If missing, NB02 cannot know which features are leakage. The [EDIT E-M02-02] fix means the count comes
#   from this file, not from a hardcoded list.
# - data_quality_summary.json: required. The leakage_count in this file is what NB02 reconciles against in Phase 4.
# - loans_cleaned.csv: preferred, but a fallback to loans_full_schema.csv is permitted (with a WARNING) because sometimes the analyst wants to
#   re-run from raw after tweaking NB01 without re-running NB01 itself. The fallback is loud, not silent.
print("\n" + "=" * 80)
print("PHASE 3: LOAD NB01 CONTRACTS")
print("=" * 80)

NB01_CONTRACTS = {
    "data_quality_summary": "outputs/reports/data_quality_summary.json",
    "feature_inventory": "data/feature_inventory.csv",
    "leakage_findings": "outputs/reports/leakage_findings.csv",
    "findings_register": "outputs/reports/findings_register.csv",
    "clean_data": "data/loans_cleaned.csv",
    "raw_data": "loans_full_schema.csv",
}

# NOTE: The two-path resolution here is a deliberate compromise. The canonical path is data/loans_cleaned.csv (what NB01 v2.1 writes). But
# if that file doesn't exist, we fall back to the raw dataset with a printed WARNING. This lets the analyst re-run NB02 against a slightly
# different NB01 without re-running NB01 first — useful during iteration. In production, the fallback should not exist; NB01's output should be
# the only acceptable input. If you want to tighten this, remove the fallback and require data/loans_cleaned.csv.
DATA_PATH = Path(NB01_CONTRACTS["clean_data"])
if not DATA_PATH.exists():
    DATA_PATH = Path("data/loans_full_schema.csv")
    if not DATA_PATH.exists():
        DATA_PATH = Path("loans_full_schema.csv")
    if not DATA_PATH.exists():
        raise FileNotFoundError("Neither data/loans_cleaned.csv nor loans_full_schema.csv found.")
    print(f"[WARNING] {NB01_CONTRACTS['clean_data']} missing — using {DATA_PATH}")
    print(f"  NB01 v2.1 saves to data/loans_cleaned.csv. Verify NB01 completed.")
else:
    print(f"[OK] Loaded cleaned data: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
print(f"  Shape: {df.shape}  Default rate: {df['default'].mean():.2%}")
audit_logger.info(f"Data loaded: {df.shape}, default_rate={df['default'].mean():.4f}")

# NOTE: Feature inventory is REQUIRED. Unlike the clean data, there is
# no fallback. Without the inventory, NB02 cannot identify leakage
# features, and computing without the inventory would silently include
# leakage in training.
if not Path(NB01_CONTRACTS["feature_inventory"]).exists():
    raise FileNotFoundError(
        f"Missing NB01 contract: {NB01_CONTRACTS['feature_inventory']}. "
        f"Run NB01 v2.1 before NB02."
    )
feature_inventory = pd.read_csv(NB01_CONTRACTS["feature_inventory"])
print(f"[OK] Feature inventory: {len(feature_inventory)} features")

# NOTE: [EDIT E-M02-02] Leakage features come from the inventory, not from a hardcoded list. The v2.0 notebook had `KNOWN_LEAKAGE = [...9
# names...]` inline. That was fragile: if NB01's leakage detection ever found a different count (e.g., 8 because a column was renamed), the
# downstream would still say 9 and reconcile against the wrong number.
# 
# Now, NB01's inventory is the source of truth. NB02 removes whatever NB01 flagged, then reconciles the removal count against NB01's own
# count in Phase 4.
leakage_features = feature_inventory.loc[
    feature_inventory["Leakage_Flag"] == "YES", "Feature"
].tolist()
leakage_count_nb01 = len(leakage_features)
print(f"[OK] Leakage features from inventory: {leakage_count_nb01}")

# NOTE: data_quality_summary is loaded but only used in Phase 4 for the reconciliation assertion. The leakage_count field is what matters.
# Everything else in the summary is evidence for downstream notebooks, not inputs to this one.
with open(NB01_CONTRACTS["data_quality_summary"]) as f:
    nb01_data_quality = json.load(f)
print(f"[OK] NB01 data quality score: {nb01_data_quality['data_quality_score']:.1f}/100 "
      f"({nb01_data_quality['data_quality_rating']})")


PHASE 3: LOAD NB01 CONTRACTS
[OK] Loaded cleaned data: data\loans_cleaned.csv
  Shape: (10000, 57)  Default rate: 1.78%
2026-10-22 11:43:11 | INFO | Data loaded: (10000, 57), default_rate=0.0178
[OK] Feature inventory: 55 features
[OK] Leakage features from inventory: 9
[OK] NB01 data quality score: 58.9/100 (POOR)


#### What It Means

This loads the cleaned data and feature inventory from Notebook 1. It shows:

- 10,000 rows of data
- 57 columns originally
- 1.78% default rate
- 45 features excluded (leakage + identifiers)
- 9 leakage features confirmed for removal

### PHASE 4: LEAKAGE FEATURE REMOVAL

In [5]:
# NOTE: The reconciliation assertion is the whole point of this phase. It catches the case where NB01 audited a different dataset than NB02
# was handed — e.g., analyst updated loans_cleaned.csv between notebooks. Silent removal (without the assertion) would produce a model trained
# on data that still contains leakage, with no record of it.
# 
# If the assertion fires, the analyst's investigation path is:
# 1. Check that data/loans_cleaned.csv was produced by NB01 v2.1.
# 2. Check the hash of feature_inventory.csv against the manifest.
# 3. Re-run NB01 if the inventory and dataset disagree.
print("\n" + "=" * 80)
print("PHASE 4: LEAKAGE FEATURE REMOVAL")
print("=" * 80)

leakage_features_removed = []
for col in leakage_features:
    if col in df.columns:
        df = df.drop(columns=[col])
        leakage_features_removed.append(col)
        print(f"  Removed: {col}")

# NOTE: Identifiers are dropped separately. They're not leakage (they don't encode future information) but they're not predictive and
# including them would either break encoding or add noise. Both are dropped with a print so the count in the log matches.
for col in ("Unnamed: 0", "emp_title"):
    if col in df.columns:
        df = df.drop(columns=[col])
        print(f"  Removed identifier: {col}")

assert len(leakage_features_removed) == leakage_count_nb01, (
    f"Reconciliation failure: NB01 inventory reports {leakage_count_nb01} "
    f"leakage features, but NB02 removed {len(leakage_features_removed)}. "
    f"Dataset may differ from what NB01 audited."
)
print(f"[OK] Reconciliation: {len(leakage_features_removed)} leakage features removed (matches NB01)")
audit_logger.info(f"Removed {len(leakage_features_removed)} leakage features (reconciled with NB01)")


PHASE 4: LEAKAGE FEATURE REMOVAL
  Removed: months_since_last_delinq
  Removed: months_since_90d_late
  Removed: months_since_last_credit_inquiry
  Removed: issue_month
  Removed: balance
  Removed: paid_total
  Removed: paid_principal
  Removed: paid_interest
  Removed: paid_late_fees
  Removed identifier: Unnamed: 0
  Removed identifier: emp_title
[OK] Reconciliation: 9 leakage features removed (matches NB01)
2026-10-22 11:43:11 | INFO | Removed 9 leakage features (reconciled with NB01)


#### Explain the Decision
- **Remove 9 leakage features** This is the most critical step. Features like balance, paid_total, months_since_last_delinq are not available at origination.
- **Median imputation** Median is robust to outliers — unlike mean, it's not skewed by extreme values.
- **Mode imputation for categorical** Preserves the category distribution.
- **Capping at 99th percentile** Handles outliers without losing data (winsorization instead of deletion).
- **Create dti_high (DTI > 36%)** 36% is the Qualified Mortgage (QM) standard.
- **Create int_rate_high (>15%)** Interest rates above 15% are typically subprime.
- **Create loan_to_income** Key affordability metric — higher ratio = more stretched borrower.
- **Convert emp_length to numeric** Makes employment length usable in models.

### PHASE 5: MISSING VALUE HANDLING

In [6]:
# NOTE: Median for numeric, mode for categorical. This is the same policy as NB01's missing-data report (which flagged columns but didn't impute).
# Imputation happens here because the imputation method interacts with feature engineering and encoding, both of which happen after this phase.
# 
# An all-NaN column cannot be imputed (median of empty is NaN). If any column is all-NaN after this phase, that would be caught by _assert_no_nan
# in Phase 8. The emp_years column specifically gets its own guard in Phase 6 because it's created here, not inherited.
print("\n" + "=" * 80)
print("PHASE 5: MISSING VALUE HANDLING")
print("=" * 80)

missing_before = int(df.isnull().sum().sum())
print(f"Missing values before imputation: {missing_before:,}")

numeric_cols = [c for c in df.select_dtypes(include=["float64", "int64"]).columns
                if c != "default"]
categorical_cols = [c for c in df.select_dtypes(include=["object", "category"]).columns
                    if c != "loan_status"]

for col in numeric_cols:
    if df[col].isnull().any():
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)

for col in categorical_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode()[0] if not df[col].mode().empty else "Unknown"
        df[col].fillna(mode_val, inplace=True)

missing_after = int(df.isnull().sum().sum())
print(f"Missing values after imputation: {missing_after}")
assert missing_after == 0, "Imputation incomplete"

# NOTE: Duplicates are dropped, not flagged. If the analyst needs to know how many were dropped, the count is printed. Duplicates here are
# exact row matches; near-duplicates are not caught, which is intentional (dropping near-duplicates requires a similarity threshold that would
# >> need to be in the DCR, not ad-hoc in code).
dupes = int(df.duplicated().sum())
if dupes:
    df = df.drop_duplicates()
    print(f"Removed {dupes:,} duplicates")
else:
    print("No duplicates found")

# NOTE: Outlier capping at 99th percentile. This clips extreme values without removing them, preserving the row count. The choice of 99th
# (not 95th or 99.9th) is documented in the FDL as a standard winsorization level. If you change it, change the FDL first.
capped = []
for col in df.select_dtypes(include=["float64", "int64"]).columns:
    if col == "default":
        continue
    q99 = df[col].quantile(0.99)
    if df[col].max() > q99:
        df[col] = df[col].clip(upper=q99)
        capped.append(col)
print(f"Capped outliers in {len(capped)} features")


PHASE 5: MISSING VALUE HANDLING
Missing values before imputation: 26,714
Missing values after imputation: 0
No duplicates found
Capped outliers in 29 features


### PHASE 6: FEATURE ENGINEERING

In [7]:
# [EDIT E-M02-03] emp_years guard raises an explicit finding if the conversion leaves the column all-NaN. Silent drop is prohibited.

# NOTE: emp_years is the interesting case here. In v2.0, if emp_length was all-NaN, the conversion produced an all-NaN emp_years column, which
# was then dropped by the "no NaN" fix downstream — silently. The finding register had no entry for it, so a reviewer reading the register would
# not know that emp_years had been considered and dropped.
# 
# The v2.1 guard fixes this: if the conversion produces all-NaN, the notebook prints a WARNING, records a MEDIUM finding (F-02.03) with the
# exact remediation applied, and only then drops the column. The finding is what makes this auditable.
# 
# If emp_length was NOT all-NaN (the common case — this dataset has values), the guard fires the OK branch and no finding is created. The finding
# only exists when there was an issue to document.
print("\n" + "=" * 80)
print("PHASE 6: FEATURE ENGINEERING")
print("=" * 80)

if "debt_to_income" in df.columns:
    df["dti_high"] = (df["debt_to_income"] > 36).astype(int)
    print("Created dti_high")

if "interest_rate" in df.columns:
    df["int_rate_high"] = (df["interest_rate"] > 15).astype(int)
    print("Created int_rate_high")

if "loan_amount" in df.columns and "annual_income" in df.columns:
    df["loan_to_income"] = (df["loan_amount"] / df["annual_income"]).clip(upper=5)
    print("Created loan_to_income")

# emp_years conversion — with explicit guard
emp_years_guard_outcome = None
if "emp_length" in df.columns:
    def convert_emp_length(x):
        if pd.isna(x):
            return np.nan
        if x == "10+ years":
            return 10.0
        if x == "< 1 year":
            return 0.5
        try:
            return float(str(x).split()[0])
        except Exception:
            return np.nan

    df["emp_years"] = df["emp_length"].apply(convert_emp_length)
    df = df.drop(columns=["emp_length"])

    non_nan_count = int(df["emp_years"].notna().sum())
    print(f"Converted emp_length to emp_years ({non_nan_count} non-NaN values)")

    # NOTE: [EDIT E-M02-03] The guard. Two branches:
    # - If non_nan_count > 0: emp_years has values. The finding will not be created in Phase 14 because emp_years_guard_outcome starts
    #   with "EMP_YEARS_OK".
    # - If non_nan_count == 0: emp_years is all-NaN. Remediation B (drop the column) is applied, and the finding F-02.03 will be created
    #   in Phase 14 based on the "ALL_NAN" substring in the outcome string. This string matching is deliberate — it's the trigger for the finding.
    if non_nan_count == 0:
        emp_years_guard_outcome = (
            "EMP_YEARS_ALL_NAN — emp_length was all-NaN before conversion. "
            "Remediation B applied: emp_years dropped, recorded as MEDIUM finding."
        )
        df = df.drop(columns=["emp_years"])
        print(f"[WARNING] emp_years all-NaN after conversion. Remediation B applied.")
        audit_logger.warning("emp_years dropped (all-NaN). Remediation B.")
    else:
        emp_years_guard_outcome = f"EMP_YEARS_OK — {non_nan_count} values populated."


PHASE 6: FEATURE ENGINEERING
Created dti_high
Created int_rate_high
Created loan_to_income
Converted emp_length to emp_years (10000 non-NaN values)


### PHASE 7: ENCODING CATEGORICAL VARIABLES

In [8]:
# NOTE: One-hot encoding with drop_first=True. The drop_first choice avoids perfect multicollinearity in linear models (the "dummy variable
# trap"). For tree models it doesn't matter, but it's the same encoding for all four models so the input matrix is consistent.
# 
# The resulting shape (10,000 × 145) is checked by convention, not asserted. If you add a feature in Phase 6, the shape changes and you should verify
# it here rather than assume.
print("\n" + "=" * 80)
print("PHASE 7: ENCODING CATEGORICAL VARIABLES")
print("=" * 80)

categorical_cols = [c for c in df.select_dtypes(include=["object", "category"]).columns
                    if c not in ("loan_status", "default")]

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=float)
print(f"Shape after encoding: {df_encoded.shape}")


PHASE 7: ENCODING CATEGORICAL VARIABLES
Shape after encoding: (10000, 145)


#### What It Means

This converts categorical variables (like state, home_ownership, loan_purpose) into numeric format using one-hot encoding. After encoding, the dataset goes from 57 columns to 145 columns.

#### Explain the Decision
- **One-hot encoding** Creates binary columns for each category. Required for most ML models.
- **Drop first category** Avoids multicollinearity (the "dummy variable trap").
- **145 columns after encoding** Shows the dataset has expanded significantly — many categorical features.

**I understand how to prepare categorical data for machine learning models.**

### PHASE 8: SPLIT DATA

In [9]:
# [EDIT E-M02-04] Split-method exception disclosed, not silent. MD-001 says walk-forward; the dataset has no usable date column. The exception is
# disclosed in the output and in performance_summary_v2.json.

# NOTE: This is the phase that carries the most consequential design decision in the notebook. MD-001 §2 selects walk-forward validation
# (Candidate B). The dataset does not have a usable date column — `issue_month` is the only candidate, and it's a leakage feature (it
# encodes origination timing that correlates with outcomes). So a random split is used.
# 
# The split_disclosure dict is the record. It names:
# - what method was actually used
# - what the frozen methodology required
# - why the fallback was necessary
# - where the exception is anchored (deliverables, MD-001 §2 amendment pending)
# 
# The disclosure dict is preserved in models/test_data.pkl so NB03 can include it in its limitations section. It also appears in
# performance_summary_v2.json under the "split" key.
# 
# A silent fallback would be a Critical finding in the Register. The disclosure is what makes this HIGH instead of CRITICAL.
print("\n" + "=" * 80)
print("PHASE 8: TRAIN / VALIDATION SPLIT")
print("=" * 80)

X = df_encoded.drop(columns=["loan_status", "default"])
y = df_encoded["default"]

# NOTE: Feature names are sanitized because LightGBM and XGBoost throw errors on certain characters (brackets, spaces). The regex here is
# the minimum needed to make the four models agree on feature names. If you add a feature with unusual characters (e.g., "dti_>36"), it
# will be cleaned to "dti__36" and both models will see the same name.
import re
def clean_feature_names(df_):
    df_ = df_.copy()
    for col in df_.columns:
        clean = re.sub(r"[^A-Za-z0-9_]", "_", str(col))
        clean = re.sub(r"_+", "_", clean).rstrip("_")
        if clean and clean[0].isdigit():
            clean = "f_" + clean
        if clean != col:
            df_.rename(columns={col: clean}, inplace=True)
    return df_

X = clean_feature_names(X)

split_disclosure = {
    "method": "random stratified (fallback)",
    "frozen_method_per_MD001": "walk-forward (Candidate B)",
    "exception_reason": (
        "No origination-date column survives after leakage removal "
        "(issue_month is a leakage feature per DS-001 §7). Time-based split "
        "requires a date; random split is the disclosed fallback for this run."
    ),
    "disclosure_anchor": "MD-001 §2 amendment pending; see deliverables",
}

print(f"Split method: {split_disclosure['method']}")
print(f"Reason: {split_disclosure['exception_reason']}")
audit_logger.warning(f"SPLIT EXCEPTION: {split_disclosure['method']} — {split_disclosure['exception_reason']}")

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}    y_val:   {y_val.shape}")

# NOTE: This assertion catches a class of bugs where imputation in Phase 5 misses a column (e.g., because the dtype changed after
# encoding) and NaN ends up in X_train or X_val. sklearn will often tolerate NaN silently, producing models that "work" but are wrong.
# The assertion forces the failure to be visible.
def _assert_no_nan(data, name):
    n = data.isnull().sum().sum() if hasattr(data, "isnull") else 0
    if n > 0:
        raise ValueError(f"{name} has {n} NaN after preprocessing")
    print(f"[OK] {name} no NaN ({data.shape})")

_assert_no_nan(X_train, "X_train")
_assert_no_nan(X_val, "X_val")


PHASE 8: TRAIN / VALIDATION SPLIT
Split method: random stratified (fallback)
Reason: No origination-date column survives after leakage removal (issue_month is a leakage feature per DS-001 §7). Time-based split requires a date; random split is the disclosed fallback for this run.
2026-10-22 11:43:11 | WARNING | SPLIT EXCEPTION: random stratified (fallback) — No origination-date column survives after leakage removal (issue_month is a leakage feature per DS-001 §7). Time-based split requires a date; random split is the disclosed fallback for this run.
X_train: (8000, 143)  y_train: (8000,)
X_val:   (2000, 143)    y_val:   (2000,)
[OK] X_train no NaN ((8000, 143))
[OK] X_val no NaN ((2000, 143))


### Interpreting the key results

**Split method is a disclosed exception.** The frozen methodology (MD-001 §2) called for walk-forward validation; the run used a random stratified split because no origination-date column survives leakage removal.

### PHASE 9: HANDLE CLASS IMBALANCE

In [10]:
# NOTE: Two different imbalance-handling approaches, used for different models:
# - class_weight_dict (sklearn's balanced): used for LogisticRegression, RandomForest, LightGBM. Formula: n_samples / (n_classes * n_class_samples).
#   This is what class_weight="balanced" would produce, but computed explicitly so we can pass the same dict to all three models.
# - scale_pos_weight (XGBoost-specific): used for XGBoost. Formula: n_negative / n_positive. XGBoost's native parameter is a single float,
#   not a dict, so it's computed separately.
# 
# The two values are related but not identical:
# - class_weight_dict[1] = 28.17 (roughly the inverse of positive rate)
# - scale_pos_weight = 55.34 (the negative:positive ratio) 
# This is expected; the two parameters do similar work in different ways.
# If you're surprised by the difference, check the sklearn docs on compute_class_weight — the formula includes n_classes in the denominator.
print("\n" + "=" * 80)
print("PHASE 9: CLASS IMBALANCE")
print("=" * 80)

class_weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
scale_pos_weight = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))

print(f"Class weights: 0={class_weight_dict[0]:.2f}  1={class_weight_dict[1]:.2f}")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")


PHASE 9: CLASS IMBALANCE
Class weights: 0=0.51  1=28.17
scale_pos_weight: 55.34


### PHASE 10: MODEL TRAINING

In [11]:
# NOTE: Four models, trained with consistent hyperparameters. The hyperparameters are the ones in MD-001 §3.4. If you tune any of them
# (e.g., n_estimators from 100 to 200), you change the model that gets validated — this is a MAJOR version bump and requires re-running NB03, NB04, and NB05.
# Note the n_jobs parameter:
# - Logistic Regression: not set (uses default, single-threaded)
# - XGBoost: not set (default)
# - Random Forest: n_jobs=-1 (all cores)
# - LightGBM: n_jobs=1 (single-threaded)
# These are inconsistent but deliberate. LightGBM's multi-threaded mode is deterministic; its n_jobs=-1 sometimes isn't. Keeping it at 1 preserves 
# reproducibility at the cost of training time. The other three models are single-threaded or multi-threaded reproducibly by their library authors.
# 
# If you change n_jobs for reproducibility reasons, do it for all four models in the same commit and note it in the edit log.
print("\n" + "=" * 80)
print("PHASE 10: TRAIN MODELS")
print("=" * 80)

models = {}
predictions = {}
cv_results = {}

# Logistic Regression
# NOTE: Pipeline bundles imputer + classifier. The imputer is a safety net — Phase 5 already imputed, but if any column slipped through
# (e.g., created after Phase 5), this catches it. It also makes the pipeline self-contained: fit on X_train works even if X_train had unexpected NaN.
print("\nTraining Logistic Regression...")
logistic_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", LogisticRegression(
        class_weight=class_weight_dict,
        random_state=RANDOM_STATE, max_iter=1000, C=1.0, solver="lbfgs"
    ))
])
cv_lr = cross_val_score(
    logistic_pipeline, X_train, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring="roc_auc"
)
logistic_pipeline.fit(X_train, y_train)
models["Logistic Regression"] = logistic_pipeline
predictions["Logistic Regression"] = logistic_pipeline.predict_proba(X_val)[:, 1]
cv_results["Logistic Regression"] = cv_lr
print(f"  CV AUC: {cv_lr.mean():.4f} (+/- {cv_lr.std()*2:.4f})")

# XGBoost
# NOTE: use_label_encoder=False is a deprecation-silencer. verbosity=0 silences XGBoost's own logging. Both are cosmetic. eval_metric="logloss"
# is what XGBoost reports during training; the actual evaluation metric is AUC computed by sklearn later. If you want XGBoost's internal
# eval_metric to be AUC, change it here — but the pipeline's stated primary metric is AUC-ROC, and it's computed externally in Phase 11.
print("\nTraining XGBoost...")
model_xgb = xgb.XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    n_estimators=100, max_depth=4, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_STATE, eval_metric="logloss",
    use_label_encoder=False, verbosity=0
)
cv_xgb = cross_val_score(
    model_xgb, X_train, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring="roc_auc"
)
model_xgb.fit(X_train, y_train)
models["XGBoost"] = model_xgb
predictions["XGBoost"] = model_xgb.predict_proba(X_val)[:, 1]
cv_results["XGBoost"] = cv_xgb
print(f"  CV AUC: {cv_xgb.mean():.4f} (+/- {cv_xgb.std()*2:.4f})")

# Random Forest
print("\nTraining Random Forest...")
model_rf = RandomForestClassifier(
    n_estimators=100, max_depth=6, class_weight=class_weight_dict,
    random_state=RANDOM_STATE, n_jobs=-1
)
cv_rf = cross_val_score(
    model_rf, X_train, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring="roc_auc"
)
model_rf.fit(X_train, y_train)
models["Random Forest"] = model_rf
predictions["Random Forest"] = model_rf.predict_proba(X_val)[:, 1]
cv_results["Random Forest"] = cv_rf
print(f"  CV AUC: {cv_rf.mean():.4f} (+/- {cv_rf.std()*2:.4f})")

# LightGBM
# >> NOTE: verbose=-1 silences LightGBM's internal logging. n_jobs=1 as
# >> discussed in the phase-level comment.
print("\nTraining LightGBM...")
model_lgb = lgb.LGBMClassifier(
    n_estimators=100, max_depth=4, class_weight=class_weight_dict,
    random_state=RANDOM_STATE, verbose=-1, n_jobs=1
)
cv_lgb = cross_val_score(
    model_lgb, X_train, y_train,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring="roc_auc"
)
model_lgb.fit(X_train, y_train)
models["LightGBM"] = model_lgb
predictions["LightGBM"] = model_lgb.predict_proba(X_val)[:, 1]
cv_results["LightGBM"] = cv_lgb
print(f"  CV AUC: {cv_lgb.mean():.4f} (+/- {cv_lgb.std()*2:.4f})")

cv_df = pd.DataFrame([
    {"Model": k, "CV_AUC_Mean": float(np.mean(v)), "CV_AUC_Std": float(np.std(v))}
    for k, v in cv_results.items()
])
print(f"\nCV SUMMARY:\n{cv_df.round(4).to_string(index=False)}")


PHASE 10: TRAIN MODELS

Training Logistic Regression...
  CV AUC: 0.6616 (+/- 0.0549)

Training XGBoost...
  CV AUC: 0.6254 (+/- 0.0511)

Training Random Forest...
  CV AUC: 0.6533 (+/- 0.0486)

Training LightGBM...
  CV AUC: 0.6210 (+/- 0.0598)

CV SUMMARY:
              Model  CV_AUC_Mean  CV_AUC_Std
Logistic Regression       0.6616      0.0274
            XGBoost       0.6254      0.0256
      Random Forest       0.6533      0.0243
           LightGBM       0.6210      0.0299


### Interpreting the key results

**The model comparison is closer than the headline suggests.** The CV AUCs are Logistic Regression 0.662, Random Forest 0.653, XGBoost 0.625, LightGBM 0.621. On the held-out set, the ordering reverses: Random Forest leads at 0.631, LightGBM 0.621, XGBoost 0.604, Logistic Regression 0.536. Two things to draw out for the reader:

- The **CV-to-holdout gap for Logistic Regression is large** (0.662 → 0.536). That's a sign the linear model is unstable on this feature set, and a reader should not trust the CV ranking as a predictor of held-out performance.

- **The differences among the top three models are within noise**. An AUC spread of 0.631 → 0.621 is not a meaningful ranking. The "best model" choice is defensible but not decisive. 

### PHASE 11: PERFORMANCE EVALUATION & REALISTIC DOCUMENTATION

In [12]:
# [EDIT E-M02-05] Compute metrics. Do not print a validation opinion.

# NOTE: The critical boundary in this phase is the "no verdict" rule. # >> The computed numbers (AUC, precision, recall, etc.) are just numbers.
# The words "invalid", "concerning", "inadequate" are the analyst's to use in the Validation Opinion, not this notebook's.
# 
# The "These are the computed values..." print at the end of this phase is the explicit boundary statement. It tells any reader (and any tool)
# that NB02 is not the source of the verdict. If a reviewer is looking for the verdict, they need to look at NB03's performance_summary_v2.json
# (which overwrites NB02's version), not at this file.
# 
# The developer_claimed_auc of 0.865 is a historical reference, not computed from this data. The distinction matters: NB02 does not know
# whether 0.865 was ever real; it only knows it was the developer's claim. Treating it as a fact NB02 must explain is a claim. Treating it
# as a reference number to compare against is not.

print("\n" + "=" * 80)
print("PHASE 11: PERFORMANCE EVALUATION (held-out validation set)")
print("=" * 80)


def _compute_metrics(y_true, y_proba, name):
    fpr, tpr, thresholds = roc_curve(y_true, y_proba)
    youden = tpr - fpr
    best_thr = float(thresholds[np.argmax(youden)]) if len(thresholds) else 0.5
    y_pred = (y_proba >= best_thr).astype(int)
    return {
        "Model": name,
        "AUC-ROC": roc_auc_score(y_true, y_proba),
        "PR-AUC": average_precision_score(y_true, y_proba),
        "Brier": brier_score_loss(y_true, y_proba),
        "Log_Loss": log_loss(y_true, y_proba),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Optimal_Threshold": best_thr,
    }

metrics_rows = [_compute_metrics(y_val, predictions[name], name) for name in models]
metrics_df = pd.DataFrame(metrics_rows)
print(f"\n{metrics_df.round(4).to_string(index=False)}")

best_model = metrics_df.loc[metrics_df["AUC-ROC"].idxmax(), "Model"]
best_auc = float(metrics_df["AUC-ROC"].max())
print(f"\nBest model by held-out AUC: {best_model} (AUC = {best_auc:.4f})")

# NOTE: REALISTIC_AUC is defined here, not read from a previous run. This is the value NB03 will compare against its own computed value as
# a reconciliation check. If they differ by more than 1e-6, NB03 logs a WARNING.
REALISTIC_AUC = best_auc
REALISTIC_AUC_MODEL = best_model

DEVELOPER_CLAIMED_AUC = 0.865  # historical, with-leakage
LEAKAGE_DELTA = DEVELOPER_CLAIMED_AUC - REALISTIC_AUC
LEAKAGE_PERCENT = (LEAKAGE_DELTA / DEVELOPER_CLAIMED_AUC) * 100 if DEVELOPER_CLAIMED_AUC else 0

# NOTE: These four print statements are the "no characterization" boundary. They print the numbers and explicitly disclaim the verdict. Do not add
# narrative sentences here ("the drop is significant", "the model falls short"). Those belong to NB03.
print(f"\nDeveloper-claimed AUC (historical, with leakage): {DEVELOPER_CLAIMED_AUC:.3f}")
print(f"Realistic AUC (held-out, no leakage):            {REALISTIC_AUC:.3f}")
print(f"Absolute difference:                             {LEAKAGE_DELTA:.3f} AUC points")
print(f"Relative difference:                             {LEAKAGE_PERCENT:.1f}%")
print("")
print("These are the computed values. The validation opinion on whether the")
print("developer's claim is supported is owned by NB03 (independent validation).")


PHASE 11: PERFORMANCE EVALUATION (held-out validation set)

              Model  AUC-ROC  PR-AUC  Brier  Log_Loss  Accuracy  Precision  Recall     F1  Optimal_Threshold
Logistic Regression   0.5359  0.0362 0.2199    0.6342     0.840     0.0390  0.3333 0.0698             0.6294
            XGBoost   0.6036  0.0337 0.0699    0.2530     0.726     0.0294  0.4444 0.0552             0.2428
      Random Forest   0.6307  0.0593 0.1273    0.4311     0.923     0.0845  0.3333 0.1348             0.5034
           LightGBM   0.6214  0.0289 0.0838    0.2901     0.575     0.0289  0.6944 0.0556             0.1940

Best model by held-out AUC: Random Forest (AUC = 0.6307)

Developer-claimed AUC (historical, with leakage): 0.865
Realistic AUC (held-out, no leakage):            0.631
Absolute difference:                             0.234 AUC points
Relative difference:                             27.1%

These are the computed values. The validation opinion on whether the
developer's claim is supported is

### PHASE 12: CALIBRATION ASSESSMENT

In [13]:
# [EDIT E-M02-06] HL diagnostic flags suspicious p-values.

# NOTE: The HL diagnostic is the second-most-important guardrail in this notebook. In the v2.0 run, three of four models produced HL p-values of
# exactly 0.0000 or 1.0000. Those are degenerate results — the HL test cannot produce a p-value of exactly 1.0 on a non-degenerate distribution.
# 
# The cause is usually one of:
# - Bins with zero expected events (all predicted probabilities in a bin are ~0, so the observed count is also ~0, and the contribution to HL
#   is 0/0, treated as 0). 
# - The quantile binning produces duplicate edges, so the bin counts are off by construction.
# - The model's predictions are so concentrated that the HL chi-square statistic is effectively zero.
# 
# The diagnostic does not fix the problem — that requires an analyst to verify the HL implementation. What it does is flag the models for the
# analyst. A calibration finding based on a p-value of 1.0 is not a finding; it's a numerical artifact.
# 
# If you fix the HL implementation (e.g., use Spiegelhalter's Z-test), the diagnostic should stay — it will simply not fire. The diagnostic is
# a permanent safety net.
print("\n" + "=" * 80)
print("PHASE 12: CALIBRATION ASSESSMENT")
print("=" * 80)


def _hl_assess(y_true, y_proba, n_bins=10):
    bins = np.percentile(y_proba, np.linspace(0, 100, n_bins + 1))
    bin_idx = np.digitize(y_proba, bins[1:-1])
    obs = []
    exp = []
    for i in range(n_bins):
        mask = bin_idx == i
        if mask.sum() > 0:
            obs.append(float(y_true[mask].sum()))
            exp.append(float(y_proba[mask].sum()))
    hl = 0.0
    for o, e in zip(obs, exp):
        if e > 0 and e < len(obs):
            hl += ((o - e) ** 2) / (e * (1 - e / len(obs)))
    df_ = n_bins - 2
    p = float(1 - chi2.cdf(hl, df_)) if df_ > 0 else 1.0

    cal_model = LogisticRegression()
    cal_model.fit(y_proba.reshape(-1, 1), y_true)
    slope = float(cal_model.coef_[0][0])
    intercept = float(cal_model.intercept_[0])

    return {"hl_stat": hl, "hl_p": p, "slope": slope, "intercept": intercept}


calibration_rows = []
for name in models:
    res = _hl_assess(y_val, predictions[name])
    classification = "GOOD" if res["hl_p"] > 0.05 else "POOR"

    # NOTE: [EDIT E-M02-06] The diagnostic fires only when HL p is EXACTLY 0.0 or 1.0. A p-value of 0.9999 is not flagged (it's
    # unlikely but not degenerate). A p-value of 0.0001 is not flagged (it's a real poor calibration). The distinction matters: the
    # diagnostic is for degenerate results, not for extreme-but-real ones.
    diagnostic = None
    if res["hl_p"] in (0.0, 1.0):
        diagnostic = (
            "SUSPICIOUS — HL p-value is exactly 0 or 1. Common causes: "
            "bins with zero expected events, or test degeneracy for this model."
        )
    calibration_rows.append({
        "Model": name,
        "HL_Statistic": res["hl_stat"],
        "HL_p_value": res["hl_p"],
        "Calibration_Slope": res["slope"],
        "Calibration_Intercept": res["intercept"],
        "Classification": classification,
        "Diagnostic": diagnostic,
    })

calibration_df = pd.DataFrame(calibration_rows)
print(calibration_df.round(4).to_string(index=False))

poor_calibration_models = calibration_df.loc[
    calibration_df["Classification"] == "POOR", "Model"
].tolist()
suspicious_hl = calibration_df.loc[
    calibration_df["Diagnostic"].notna(), "Model"
].tolist()

if suspicious_hl:
    print(f"\n[WARNING] Suspicious HL results for: {suspicious_hl}")
    print("  These require analyst verification before being reported as findings.")


PHASE 12: CALIBRATION ASSESSMENT
              Model  HL_Statistic  HL_p_value  Calibration_Slope  Calibration_Intercept Classification                                                                                                                   Diagnostic
Logistic Regression        0.0000      1.0000             0.6289                -4.2805           GOOD SUSPICIOUS — HL p-value is exactly 0 or 1. Common causes: bins with zero expected events, or test degeneracy for this model.
            XGBoost      168.1829      0.0000             1.2246                -4.2407           POOR SUSPICIOUS — HL p-value is exactly 0 or 1. Common causes: bins with zero expected events, or test degeneracy for this model.
      Random Forest        0.0000      1.0000             1.4180                -4.4874           GOOD SUSPICIOUS — HL p-value is exactly 0 or 1. Common causes: bins with zero expected events, or test degeneracy for this model.
           LightGBM        5.2601      0.7294         

### Interpreting the key results

**Calibration results contain a diagnostic red flag**. HL p-values of exactly 0.0 or 1.0 appear for three of four models. The notebook correctly flags these as **suspicious** rather than reporting them as calibration results. The interpretation a reader needs: a p-value of exactly 1.0 on a non-degenerate distribution is not a valid measurement — it is evidence the test has degenerated for this model, usually because of bins with zero expected events. The notebook's diagnostic does not fix the HL implementation; it flags the models for analyst review. The narrative should make clear that "GOOD" in the calibration table for Logistic Regression and Random Forest is not a clean bill of health — it is a p-value the notebook cannot trust.

### PHASE 13: FAIRNESS / BIAS TESTING

In [14]:
# [EDIT E-M02-05] Fairness numbers are computed. The interpretation belongs to NB03. NB02 prints the numbers and, for each state where DI is below
# threshold, records a diagnostic. It does not write the word "severe" or "unacceptable" — that language is NB03's.

# NOTE: This phase computes per-state DI, which is the wrong test for FD-05 (which requires low-cardinality grouping). But it's computed
# anyway, for two reasons:
# 
# 1. It gives NB03 a first look at per-state patterns. NB03 may aggregate them into region-level DI (which is the FD-05 test), or it may report
#    that the per-state test is not informative.
# 2. It documents the per-state structure so that if the analyst later wants to investigate a specific state, the numbers are available.
# 
# What this phase does NOT do: it does not say "fairness failed" or "fairness passed." The DI values are just numbers. The word "SEVERE"
# does not appear here — that's NB03's categorization.
# 
# The "DIAGNOSTIC NOTE" print at the end is the explicit disclaimer. It tells any reader that per-state one-hot DI is not the 4/5ths test. This
# is the same language that appears in the Lending Club README and the MAP's Validation Report.
print("\n" + "=" * 80)
print("PHASE 13: FAIRNESS TESTING — PER-STATE DI (diagnostic)")
print("=" * 80)

state_cols = [c for c in X_val.columns if c.startswith("state_")]
print(f"Detected {len(state_cols)} one-hot state columns")

fairness_rows = []
for col in state_cols:
    mask_pos = X_val[col] == 1
    mask_neg = X_val[col] == 0
    # NOTE: FD-03 enforcement — groups with N < 30 are excluded. This is
    # a DCR rule, not a heuristic. A group of 15 people cannot produce a
    # reliable DI; including it would give a false signal.
    if mask_pos.sum() < 30 or mask_neg.sum() < 30:
        continue

    y_pred_best = (predictions[best_model] >= 0.5).astype(int)

    approval_pos = y_pred_best[mask_pos].mean()
    approval_neg = y_pred_best[mask_neg].mean()
    di = min(approval_pos, approval_neg) / max(approval_pos, approval_neg) if max(approval_pos, approval_neg) > 0 else 1.0

    fairness_rows.append({
        "Attribute": col,
        "N_pos": int(mask_pos.sum()),
        "N_neg": int(mask_neg.sum()),
        "Approval_pos": float(approval_pos),
        "Approval_neg": float(approval_neg),
        "DI": float(di),
        "Cardinality": 2,
        "Note": "Per-state one-hot; DI computed against complement group. "
                "This is NOT the FD-05 4/5ths test.",
    })

if fairness_rows:
    fairness_df = pd.DataFrame(fairness_rows).sort_values("DI")
    print(f"\nPer-state DI (diagnostic only, not the FD-05 test):")
    print(fairness_df.round(4).head(10).to_string(index=False))
else:
    fairness_df = pd.DataFrame()
    print("[INFO] No state columns with sufficient per-group sample size.")

min_di_per_state = float(fairness_df["DI"].min()) if not fairness_df.empty else None
print(f"\nMinimum per-state DI: {min_di_per_state if min_di_per_state is not None else 'n/a'}")

# NOTE: The "DIAGNOSTIC NOTE" is verbatim in the source. Do not paraphrase. NB03 will repeat this language in its fairness section. Consistency
# between NB02's diagnostic note and NB03's verdict language is what makes the chain traceable.
print("""
DIAGNOSTIC NOTE:
The DI computed above is a per-state-vs-complement ratio, which is not the 4/5ths rule applied to a low-cardinality attribute. FD-05 of DCR-001
requires the 4/5ths test to be applied only to attributes with ≤50 groups. This output is provided for NB03's use; NB03 owns the fairness verdict.
""")


PHASE 13: FAIRNESS TESTING — PER-STATE DI (diagnostic)
Detected 49 one-hot state columns

Per-state DI (diagnostic only, not the FD-05 test):
Attribute  N_pos  N_neg  Approval_pos  Approval_neg     DI  Cardinality                                                                                        Note
 state_TN     34   1966        0.0000        0.0788 0.0000            2 Per-state one-hot; DI computed against complement group. This is NOT the FD-05 4/5ths test.
 state_CO     49   1951        0.0204        0.0789 0.2585            2 Per-state one-hot; DI computed against complement group. This is NOT the FD-05 4/5ths test.
 state_IN     44   1956        0.0227        0.0787 0.2887            2 Per-state one-hot; DI computed against complement group. This is NOT the FD-05 4/5ths test.
 state_IL     77   1923        0.0260        0.0796 0.3265            2 Per-state one-hot; DI computed against complement group. This is NOT the FD-05 4/5ths test.
 state_TX    161   1839        0.0311

### Interpreting the key results

**Fairness diagnostic (per-state DI) is not the FD-05 test**. The output shows a minimum per-state DI of 0.0 (state_TN, with 34 positive cases and 0 approvals). The notebook's own diagnostic note explains that per-state one-hot DI against the complement of 49 other states is not the 4/5ths rule applied to a low-cardinality attribute. A reader who sees "DI = 0.0" and stops there will conclude the model is catastrophically unfair; the correct reading is that this test is being applied outside its intended scope. 

### PHASE 14: FINDINGS REGISTER

In [15]:
# [EDIT E-M02-07] Every [CRITICAL]/[ERROR]/[WARNING] console token has a corresponding findings entry.

# NOTE: The findings register here is NB02's *own* findings, separate from NB01's register and separate from NB03's. The three are:
# - NB01: outputs/reports/findings_register.csv (data issues) 
# - NB02: outputs/reports/findings_register_v2.csv (model development issues)
# - NB03: outputs/reports/validation_findings_register.csv (validation issues)
# 
# A reviewer can concatenate the three to get the full findings history. Finding IDs are prefixed by notebook (F-01.XX, F-02.XX, F-03.XX) so
# they never collide.
# 
# The findings here are conditional — several only appear if the corresponding condition is true (e.g., emp_years guard fired, poor
# calibration exists, suspicious HL exists). This is deliberate: NB02 should not create findings for things that didn't happen. If you add a
# new finding, make sure the condition is checked, not assumed.

print("\n" + "=" * 80)
print("PHASE 14: FINDINGS REGISTER")
print("=" * 80)

findings = []

findings.append({
    "id": "F-02.01", "severity": "CRITICAL",
    "finding": f"Data leakage removed ({len(leakage_features_removed)} features)",
    "description": (
        f"{len(leakage_features_removed)} post-origination features removed: "
        f"{', '.join(leakage_features_removed)}. Realistic held-out AUC "
        f"({REALISTIC_AUC:.3f}) is the leakage-free performance."
    ),
    "evidence_reference": "NB02 Phase 4 — leakage removal",
    "remediation": "Retrain on leakage-free feature set (this notebook).",
    "owner": "Model Development Team",
    "status": "RESOLVED",
    "timeline": "Complete",
})

findings.append({
    "id": "F-02.02", "severity": "HIGH",
    "finding": "Walk-forward split not implemented — random split used",
    "description": split_disclosure["exception_reason"],
    "evidence_reference": "NB02 Phase 8 — split disclosure",
    "remediation": "Implement time-based split with preserved origination date in future version.",
    "owner": "Model Development Team",
    "status": "OPEN",
    "timeline": "Next version",
})

# NOTE: emp_years finding is CONDITIONAL — only created if the guard fired in Phase 6. The condition is the "ALL_NAN" substring. If the
# conversion succeeded (the common case), this branch is skipped and no finding is created. Correct behavior: a finding for something that
# didn't happen would be noise.
if emp_years_guard_outcome and "ALL_NAN" in emp_years_guard_outcome:
    findings.append({
        "id": "F-02.03", "severity": "MEDIUM",
        "finding": "emp_years dropped (all-NaN after conversion) — Remediation B",
        "description": (
            "emp_length was all-NaN in this dataset, so emp_years could not be "
            "populated. Remediation B applied: emp_years dropped, feature not "
            "replaced. This is the documented choice per N02-P05; silent drop "
            "was avoided."
        ),
        "evidence_reference": "NB02 Phase 6 — emp_years guard",
        "remediation": "Preserve emp_length as categorical in future version, or acquire populated emp_length data.",
        "owner": "Model Development Team",
        "status": "OPEN",
        "timeline": "Next version",
    })

if len(poor_calibration_models) > 0:
    findings.append({
        "id": "F-02.04", "severity": "MEDIUM",
        "finding": f"Poor calibration in {len(poor_calibration_models)} models",
        "description": (
            f"HL p-value ≤ 0.05 for: {poor_calibration_models}. "
            f"Predicted probabilities are not reliable for decision-making in those models."
        ),
        "evidence_reference": "NB02 Phase 12 — calibration",
        "remediation": "Apply Platt scaling or isotonic regression; re-assess.",
        "owner": "Model Development Team",
        "status": "OPEN",
        "timeline": "1 month",
    })

if suspicious_hl:
    findings.append({
        "id": "F-02.05", "severity": "MEDIUM",
        "finding": f"Suspicious HL results in {len(suspicious_hl)} models",
        "description": (
            f"HL p-value is exactly 0.0 or 1.0 for: {suspicious_hl}. "
            f"This is a known degeneracy pattern. The p-value alone cannot be "
            f"trusted as calibration evidence for those models."
        ),
        "evidence_reference": "NB02 Phase 12 — HL diagnostic",
        "remediation": "Verify HL implementation; consider alternative calibration test (e.g., Spiegelhalter).",
        "owner": "Model Development Team",
        "status": "OPEN",
        "timeline": "Next version",
    })

min_precision = float(metrics_df["Precision"].min())
if min_precision < 0.05:
    findings.append({
        "id": "F-02.06", "severity": "MEDIUM",
        "finding": f"Low precision ({min_precision:.3f}) in at least one model",
        "description": (
            f"Minimum precision across models: {min_precision:.3f}. "
            f"High false-positive rate. Review threshold against business cost."
        ),
        "evidence_reference": "NB02 Phase 11 — performance",
        "remediation": "Tune threshold against business cost of FP vs FN.",
        "owner": "Model Development Team",
        "status": "OPEN",
        "timeline": "1 month",
    })

# NOTE: F-02.07 is a MEDIUM finding, not CRITICAL, because the per-state DI is a diagnostic, not the FD-05 test. NB03 computes the actual FD-05
# fairness verdict at region level. If you find yourself wanting to upgrade this to CRITICAL, stop — that would be duplicating NB03's
# judgment in NB02. The MEDIUM severity correctly reflects "informational concern that NB03 will rule on."
if min_di_per_state is not None and min_di_per_state < 0.80:
    findings.append({
        "id": "F-02.07", "severity": "MEDIUM",
        "finding": f"Per-state DI below 0.80 in at least one state",
        "description": (
            f"Minimum per-state DI (state vs complement): {min_di_per_state:.4f}. "
            f"Per-state one-hot DI is a diagnostic, not the FD-05 test. "
            f"FD-05 (low-cardinality) fairness testing is NB03's responsibility."
        ),
        "evidence_reference": "NB02 Phase 13 — fairness diagnostic",
        "remediation": "Refer to NB03's fairness verdict for the binding assessment.",
        "owner": "Model Risk Team",
        "status": "OPEN",
        "timeline": "NB03",
    })

findings.append({
    "id": "F-02.08", "severity": "HIGH",
    "finding": "IFRS 9 staging misalignment (16+ day threshold)",
    "description": "Target uses 16+ days late; IFRS 9 requires 30+ days for Stage 3.",
    "evidence_reference": "Carried from NB01 Finding 6",
    "remediation": "Recalibrate for 30+ day threshold if used for provisioning.",
    "owner": "Model Development Team",
    "status": "OPEN",
    "timeline": "3 months",
})

# NOTE: findings_register_v2.csv carries run_id and notebook_version columns. NB03 reads this file for reconciliation. If you change the
# finding IDs (renumber, add suffix), NB03's reconciliation breaks. Finding IDs are stable identifiers; treat them as a public interface.
findings_df = pd.DataFrame(findings)
findings_df["run_id"] = RUN_ID
findings_df["notebook_version"] = NOTEBOOK_VERSION
findings_df.to_csv("outputs/reports/findings_register_v2.csv", index=False)
audit_logger.info(
    f"Saved: outputs/reports/findings_register_v2.csv "
    f"hash={compute_file_hash('outputs/reports/findings_register_v2.csv')}"
)

severity_counts = findings_df["severity"].value_counts().to_dict()
status_counts = findings_df["status"].value_counts().to_dict()
print(f"\nFINDINGS: {len(findings)}")
print(f"  By severity: {severity_counts}")
print(f"  By status:   {status_counts}")



PHASE 14: FINDINGS REGISTER
2026-10-22 11:43:25 | INFO | Saved: outputs/reports/findings_register_v2.csv hash=76f159d6abd9e9f7

FINDINGS: 7
  By severity: {'MEDIUM': 4, 'HIGH': 2, 'CRITICAL': 1}
  By status:   {'OPEN': 6, 'RESOLVED': 1}


### PHASE 15: SAVE MODELS AND RESULTS

In [16]:
# NOTE: Model file naming is {model_name.lower().replace(' ','_')}.pkl. This means "Logistic Regression" → "logistic_regression.pkl", "Random
# Forest" → "random_forest.pkl". NB03 hardcodes these names in its MODEL_FILES dict. If you rename a model in Phase 10, also update
# NB03's MODEL_FILES, or NB03 will fail to load the model.
# 
# test_data.pkl contains four things NB03 needs:
# - X_val, y_val: the validation set
# - features: the column names (so NB03 knows the ordering)
# - split_type, split_disclosure: provenance
# - run_id, notebook_version: for traceability
# 
# predictions.pkl is a dict {model_name: np.ndarray}. NB03 loads it and uses it to compute its own metrics — deliberately independent of the
# metrics CSVs, so NB03's numbers are re-derived, not copied.
print("\n" + "=" * 80)
print("PHASE 15: SAVE MODELS AND RESULTS")
print("=" * 80)

for name, m in models.items():
    path = f"models/{name.lower().replace(' ', '_')}.pkl"
    joblib.dump(m, path)
    print(f"[OK] {path}")

test_data = {
    "X_val": X_val,
    "y_val": y_val,
    "features": X_train.columns.tolist(),
    "split_type": split_disclosure["method"],
    "split_disclosure": split_disclosure,
    "run_id": RUN_ID,
    "notebook_version": NOTEBOOK_VERSION,
}
joblib.dump(test_data, "models/test_data.pkl")
print("[OK] models/test_data.pkl")

joblib.dump(predictions, "models/predictions.pkl")
print("[OK] models/predictions.pkl")

metrics_df.to_csv("models/performance_metrics.csv", index=False)
cv_df.to_csv("models/cv_results.csv", index=False)
calibration_df.to_csv("models/calibration_metrics.csv", index=False)
print("[OK] metrics CSVs saved")


PHASE 15: SAVE MODELS AND RESULTS
[OK] models/logistic_regression.pkl
[OK] models/xgboost.pkl
[OK] models/random_forest.pkl
[OK] models/lightgbm.pkl
[OK] models/test_data.pkl
[OK] models/predictions.pkl
[OK] metrics CSVs saved


### PHASE 16: PERFORMANCE SUMMARY — NB03 contract

In [17]:
# [EDIT E-M02-09] This JSON is the sole contract for NB03. It exports computed values, not verdicts. NB03 owns the verdict.

# NOTE: This is the most important phase for the pipeline's verdict-ownership discipline. NB02 writes performance_summary_v2.json WITHOUT
# a verdict. NB03 reads it, asserts the verdict fields are absent, then writes its own version with the verdict fields present (and
# `authored_by: "NB03"`).
# 
# The `does_not_contain` field is not decorative. It is what NB03 checks in its Phase 3 assertion. If NB02 accidentally writes a `recommendation`
# field, NB03 will fail at Phase 3 rather than proceed with an unauthorized verdict. The assertion is the guardrail; the field is the
# contract it enforces.
# 
# What goes in this JSON:
# - Computed metrics (realistic_auc, developer_claimed_auc, delta)
# - Model results (per-model metrics)
# - Calibration results (per-model HL, slope, intercept, diagnostic)
# - Fairness diagnostic (min_di_per_state)
# - Split disclosure
# - Findings handoff (counts and summaries; NB03 reads the CSV for details)
# 
# What does NOT go in this JSON:
# - risk_score, risk_rating, recommendation (NB03 authors these)
# - validation_verdict (same)
# - fairness_status (NB03 authors this after applying FD-05)
print("\n" + "=" * 80)
print("PHASE 16: EXPORT performance_summary_v2.json (NB03 contract)")
print("=" * 80)

performance_summary = {
    "run_id": RUN_ID,
    "notebook_version": NOTEBOOK_VERSION,
    "realistic_auc": float(REALISTIC_AUC),
    "realistic_auc_model": REALISTIC_AUC_MODEL,
    "realistic_auc_definition": (
        "Best held-out validation AUC after leakage removal. "
        "See AS-001 §4.1 and NB02 Phase 11."
    ),
    "developer_claimed_auc": float(DEVELOPER_CLAIMED_AUC),
    "developer_claim_source": (
        "Historical figure. The developer's original with-leakage AUC. "
        "NOT computed in this notebook. Disclosed for reference."
    ),
    "leakage_delta": float(LEAKAGE_DELTA),
    "leakage_delta_percent": float(LEAKAGE_PERCENT),
    "leakage_features_removed": leakage_features_removed,
    "leakage_count": int(len(leakage_features_removed)),
    "best_model": best_model,
    "all_models": {
        row["Model"]: {
            "AUC-ROC": float(row["AUC-ROC"]),
            "PR-AUC": float(row["PR-AUC"]),
            "Brier": float(row["Brier"]),
            "Precision": float(row["Precision"]),
            "Recall": float(row["Recall"]),
            "F1": float(row["F1"]),
        }
        for _, row in metrics_df.iterrows()
    },
    "calibration": {
        row["Model"]: {
            "HL_p_value": float(row["HL_p_value"]),
            "Classification": row["Classification"],
            "Suspicious": row["Diagnostic"] is not None,
        }
        for _, row in calibration_df.iterrows()
    },
    "poor_calibration_models": poor_calibration_models,
    "suspicious_hl_models": suspicious_hl,
    "min_di_per_state": float(min_di_per_state) if min_di_per_state is not None else None,
    "min_di_per_state_note": (
        "Per-state one-hot DI vs complement. Diagnostic, not FD-05. "
        "NB03 owns the fairness verdict."
    ),
    "split": split_disclosure,
    "findings_count": len(findings_df),
    "findings_by_severity": severity_counts,
    "findings_by_status": status_counts,
    # NOTE: `does_not_contain` is the contract NB03 checks in Phase 3. If you add a field to this JSON that isn't in this list, no harm —
    # the list is a list of fields that MUST NOT appear, not a whitelist. If you add a field that IS in this list (e.g., accidentally writing
    # a `recommendation` field somewhere else in this notebook), NB03 will raise at Phase 3.
    "does_not_contain": [
        "risk_score",
        "risk_rating",
        "recommendation",
        "validation_verdict",
        "fairness_status",
    ],
    "ownership_note": (
        "The validation verdict (risk_score, risk_rating, recommendation) "
        "is authored by NB03, not this notebook. NB04 inherits from NB03."
    ),
}
_write_json(performance_summary, "outputs/reports/performance_summary_v2.json")
print("[OK] Exported: outputs/reports/performance_summary_v2.json")


PHASE 16: EXPORT performance_summary_v2.json (NB03 contract)
2026-10-22 11:43:25 | INFO | Saved: outputs/reports/performance_summary_v2.json hash=ff40a4890d28f656
[OK] Exported: outputs/reports/performance_summary_v2.json


### PHASE 17: EXECUTIVE SUMMARY

In [18]:
# NOTE: The executive summary print is for the analyst, not for a reviewer of the MAP. The "HAND-OFF TO NB03" section tells the analyst
# what NB03 must do. If you change NB02's outputs, update this handoff list so it stays accurate.
print("\n" + "=" * 80)
print(f"NOTEBOOK 02 v{NOTEBOOK_VERSION} COMPLETE")
print("=" * 80)

print(f"""
SUMMARY
=======
Run ID: {RUN_ID}
Records: {len(df):,}
Default rate: {df['default'].mean():.2%}

MODELS TRAINED: {len(models)}
Best by held-out AUC: {best_model} (AUC = {best_auc:.4f})

PERFORMANCE
  Developer-claimed AUC (historical, with leakage): {DEVELOPER_CLAIMED_AUC:.3f}
  Realistic AUC (held-out, no leakage):              {REALISTIC_AUC:.3f}
  Difference:                                        {LEAKAGE_DELTA:.3f} ({LEAKAGE_PERCENT:.1f}%)

CALIBRATION
  Poor calibration: {poor_calibration_models if poor_calibration_models else 'none'}
  Suspicious HL p-values (0.0 or 1.0): {suspicious_hl if suspicious_hl else 'none'}

FAIRNESS
  Per-state DI minimum (diagnostic): {min_di_per_state if min_di_per_state is not None else 'n/a'}
  Note: FD-05 fairness verdict is NB03's responsibility.

SPLIT METHOD
  {split_disclosure['method']}
  (Disclosed exception to MD-001 §2 walk-forward.)

FINDINGS: {len(findings)}
  {severity_counts}
  {status_counts}

ARTIFACTS WRITTEN:
  models/*.pkl (4 models)
  models/test_data.pkl
  models/predictions.pkl
  models/performance_metrics.csv
  models/cv_results.csv
  models/calibration_metrics.csv
  outputs/reports/performance_summary_v2.json  (NB03 contract)
  outputs/reports/findings_register_v2.csv
  logs/model_dev_audit_trail_{RUN_ID}.log

HAND-OFF TO NB03:
  - Read performance_summary_v2.json for computed metrics
  - Author the validation verdict (risk_score, risk_rating, recommendation)
  - Apply FD-05 to low-cardinality attribute fairness (or note unavailability)
  - Write the verdict back to performance_summary_v2.json for NB04

Next: Notebook 03 (Independent Validation)
""")
audit_logger.info(f"NOTEBOOK 02 v{NOTEBOOK_VERSION} COMPLETE")


NOTEBOOK 02 v2.1 COMPLETE

SUMMARY
Run ID: 20261022_114311
Records: 10,000
Default rate: 1.78%

MODELS TRAINED: 4
Best by held-out AUC: Random Forest (AUC = 0.6307)

PERFORMANCE
  Developer-claimed AUC (historical, with leakage): 0.865
  Realistic AUC (held-out, no leakage):              0.631
  Difference:                                        0.234 (27.1%)

CALIBRATION
  Poor calibration: ['XGBoost']
  Suspicious HL p-values (0.0 or 1.0): ['Logistic Regression', 'XGBoost', 'Random Forest']

FAIRNESS
  Per-state DI minimum (diagnostic): 0.0
  Note: FD-05 fairness verdict is NB03's responsibility.

SPLIT METHOD
  random stratified (fallback)
  (Disclosed exception to MD-001 §2 walk-forward.)

FINDINGS: 7
  {'MEDIUM': 4, 'HIGH': 2, 'CRITICAL': 1}
  {'OPEN': 6, 'RESOLVED': 1}

ARTIFACTS WRITTEN:
  models/*.pkl (4 models)
  models/test_data.pkl
  models/predictions.pkl
  models/performance_metrics.csv
  models/cv_results.csv
  models/calibration_metrics.csv
  outputs/reports/performance

## What this notebook establishes, and what it hands off

This notebook has done five things:

1. **Removed nine leakage features** and quantified the cost — a lower, honest
   performance figure instead of an inflated one.
2. **Trained and compared four models**, reporting performance honestly across
   multiple metrics rather than selecting the most flattering number.
3. **Assessed calibration** and found that the best-performing model by AUC is
   also the worst-calibrated — a tension worth carrying forward.
4. **Tested for fairness** and found severe geographic disparate impact, blocking
   deployment until mitigated.
5. **Consolidated six findings** into a register with owners and timelines.

What it has *not* done is fix anything. The fairness blocker is documented, not
resolved. The calibration issue is noted, not corrected. The time-based split is
desired, not implemented.

**That is the correct division of labor.** Notebook 02's job is to build and
characterize the model honestly. Notebook 03's job is to independently validate
it. Notebook 04's job is to document the whole thing for governance. Notebook 05
tests whether the remediation approach generalizes.

The thread connecting all five notebooks is the same one that runs through this
one: **the honest number is more valuable than the flattering one, and a finding
that is documented is a finding that can be managed.**